# Exercício: Classificação Multiclasse

## 1. Crie um arquivo Jupyter Notebook e realize as seguintes operações:

a. Ler o dataset fakeTelegram.BR_2022.csv

b. Remova os trava-zaps, as linhas repetidas (duplicadas) e textos com menos de 5 palavras.

c. Agrupe as linhas com postagens iguais ou extremamente semelhantes. Aqui você pode utilizar uma métrica de semelhança de textos. Crie uma variável para representar a quantidade de vezes que a mensagem foi compartilhada. Observe que ao agrupar linhas que possuem a “mesma” postagem (texto), você deve escolher como valor para as variáveis data e hora da postagem, os valores da cópia mais antiga.

d. Você pode criar novos atributos numéricos, tais como: quantidade de palavras, quantidade de caracteres etc.

In [ ]:
import duckdb
import unicodedata
import re

import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed
from scipy.sparse import hstack
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from datasketch import MinHash, MinHashLSH


In [ ]:
conn = duckdb.connect()

telegram_data = conn.read_csv("../data/fakeTelegram.BR_2022.csv")

query = """
    SELECT * FROM telegram_data
"""

df = conn.execute(query).fetchdf()

In [2]:
df.head()

,date_message,id_member_anonymous,id_group_anonymous,media,media_type,media_url,has_media,has_media_url,trava_zap,text_content_anonymous,dataset_info_id,date_system,score_sentiment,score_misinformation,id_message,message_type,messenger,media_name,media_md5
0,2022-10-05 06:25:04,1078cc958f0febe28f4d03207660715f,12283e08a2eb5789201e105b34489ee7,None,None,None,False,False,False,Então é Fato Renato o áudio que eu ouvi no wha...,5,2022-10-05 06:25:28.863641,0.0000,NaN,16385,Texto,telegram,None,None
1,2022-10-05 06:25:08,None,12283e08a2eb5789201e105b34489ee7,None,None,None,False,False,False,"Saiu no YouTube do presidente a 8 horas atrás,...",5,2022-10-05 06:25:28.926311,0.0644,NaN,16386,Texto,telegram,None,None
2,2022-10-05 06:26:28,92a2d8fd7144074f659d1d29dc3751da,9f2d7394334eb224c061c9740b5748fc,None,None,None,False,False,False,"É isso, nossa parte já foi quase toda feita. N...",5,2022-10-05 06:26:29.361949,-0.3551,0.157242,16366,Texto,telegram,None,None
3,2022-10-05 06:27:28,d60aa38f62b4977426b70944af4aff72,c8f2de56550ed0bf85249608b7ead93d,94dca4cda503100ebfda7ce2bcc060eb.jpg,image/jpg,None,True,False,False,GENTE ACHEI ELES EM UMA SEITA MAÇONÁRICA,5,2022-10-05 06:27:29.935624,0.0000,NaN,19281,Imagem,telegram,None,94dca4cda503100ebfda7ce2bcc060eb
4,2022-10-05 06:27:44,cd6979b0b5265f08468fa1689b6300ce,e56ec342fc599ebb4ed89655eb6f03aa,5ad5c8bbe9da93a37fecf3e5aa5b0637.jpg,image/jpg,None,True,False,False,None,5,2022-10-05 06:28:29.316325,NaN,NaN,507185,Imagem,telegram,None,5ad5c8bbe9da93a37fecf3e5aa5b0637


In [3]:
df.shape

(557586, 19)

In [4]:
df = conn.execute(f"""
    SELECT * 
    FROM telegram_data
    WHERE trava_zap IS NOT TRUE
""").fetch_df()

In [5]:
df.shape

(557570, 19)

In [6]:
df_ = conn.execute("SELECT DISTINCT * FROM df").fetch_df()

In [7]:
query = """
SELECT *
FROM df_
WHERE array_length(string_split(text_content_anonymous, ' ')) >= 5
"""

df = conn.execute(query).fetch_df()

df.shape

(336944, 19)

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Converte para minúsculas
    text = text.lower()
    
    # 2. Remover URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Remover caracteres árabes
    text = re.sub(r'[\u0600-\u06FF]+', '', text)
    
    # 4. Remover acentos (Normalização)
    text = unicodedata.normalize('NFKD', text)\
           .encode('ascii', 'ignore')\
           .decode('utf-8')
    
    # 5. Remover pontuação e caracteres especiais
    text = re.sub(r'[^\w\s]', '', text)
    
    # 6. Remover emojis 
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # símbolos & pictogramas
        u"\U0001F680-\U0001F6FF"  # transporte & símbolos
        u"\U0001F1E0-\U0001F1FF"  # bandeiras (iOS)
                           "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # 7. Remover espaços extras (no início, fim e múltiplos espaços no meio)
    text = ' '.join(text.split())
    
    return text

df['cleaned_text'] = df['text_content_anonymous'].apply(clean_text)

In [ ]:
def get_shingles(text, k=5):
    """Cria um conjunto de shingles de tamanho k a partir de um texto."""
    if not isinstance(text, str) or len(text) < k:
        return set()
    return set(text[i:i+k] for i in range(len(text) - k + 1))

def create_minhash(text, num_perm, shingle_size):
    mh = MinHash(num_perm=num_perm)
    shingles = get_shingles(text, k=shingle_size)
    if not shingles:
        return None
    for shingle in shingles:
        mh.update(shingle.encode('utf-8'))
    return mh

In [10]:
print("Iniciando agrupamento de textos semelhantes...")

JACCARD_THRESHOLD = 0.7
NUM_PERMUTATIONS = 128
SHINGLE_SIZE = 5

texts = df['cleaned_text'].tolist()

print("Etapa 1 de 4: Criando MinHashes em paralelo...")
minhash_list = Parallel(n_jobs=-1)(
    delayed(create_minhash)(text, NUM_PERMUTATIONS, SHINGLE_SIZE) 
    for text in tqdm(texts, desc="Criando assinaturas")
)

# Remapeia a lista para o dicionário, o que é rápido
minhashes = {idx: mh for idx, mh in enumerate(minhash_list) if mh is not None}

# Progresso na indexação no LSH
print("\nEtapa 2 de 4: Indexando MinHashes no LSH...")
lsh = MinHashLSH(threshold=JACCARD_THRESHOLD, num_perm=NUM_PERMUTATIONS)
for idx, mh in tqdm(minhashes.items(), desc="Indexando no LSH"):
    lsh.insert(idx, mh)

# Progresso na consulta e agrupamento
print("\nEtapa 3 de 4: Consultando LSH e agrupando...")
groups = []
used = set()
for idx in tqdm(minhashes, desc="Consultando e agrupando"):
    if idx not in used:
        similar_indices = lsh.query(minhashes[idx])
        groups.append(similar_indices)
        used.update(similar_indices)

print(f"\nAgrupamento concluído. Foram encontrados {len(groups)} grupos de mensagens.")


print("\nEtapa 4 de 4: Consolidando resultados...")
grouped_data = []
for group in tqdm(groups, desc="Consolidando grupos"):
    group_df = df.iloc[list(group)]
    oldest = group_df.loc[group_df['date_message'].idxmin()]
    count = len(group_df)
    new_row = oldest.to_dict()
    new_row['shared_count'] = count
    grouped_data.append(new_row)

result_df = pd.DataFrame(grouped_data)
result_df = result_df.sort_values(by='shared_count', ascending=False)

print("\nProcesso finalizado com sucesso!")

Iniciando agrupamento de textos semelhantes...
Etapa 1 de 4: Criando MinHashes em paralelo...


Criando assinaturas: 100%|██████████| 336944/336944 [02:31<00:00, 2224.48it/s]



Etapa 2 de 4: Indexando MinHashes no LSH...


Indexando no LSH: 100%|██████████| 336652/336652 [00:11<00:00, 28843.55it/s]



Etapa 3 de 4: Consultando LSH e agrupando...


Consultando e agrupando: 100%|██████████| 336652/336652 [00:04<00:00, 74537.23it/s]



Agrupamento concluído. Foram encontrados 183796 grupos de mensagens.

Etapa 4 de 4: Consolidando resultados...


Consolidando grupos: 100%|██████████| 183796/183796 [01:02<00:00, 2933.29it/s]



Processo finalizado com sucesso!


In [11]:
def extract_features(text):
    if not isinstance(text, str):
        return {
            'word_count': 0,
            'char_count': 0,
            'unique_word_ratio': 0,
            'avg_word_length': 0
        }
    
    words = text.split()
    word_count = len(words)
    char_count = len(text)
    
    if word_count > 0:
        unique_words = set(words)
        unique_word_ratio = len(unique_words) / word_count
        avg_word_length = sum(len(word) for word in words) / word_count
    else:
        unique_word_ratio = 0
        avg_word_length = 0
    
    return {
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_ratio': unique_word_ratio,
        'avg_word_length': avg_word_length
    }

features = result_df['cleaned_text'].apply(lambda x: pd.Series(extract_features(x)))

result_df = pd.concat([result_df, features], axis=1)

print("Características numéricas adicionadas com sucesso ao result_df.")
print(result_df[['cleaned_text', 'shared_count', 'word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']].head())

Características numéricas adicionadas com sucesso ao result_df.
                                           cleaned_text  shared_count  \
1628  this community was blocked in brazil following...         17422   
1528  welcome helen user professional tool for manag...          7359   
1512  welcome helen user professional tool for manag...          3426   
1540  bm pressione o botao abaixo dentro de 5 minuto...          2649   
77    ola gisele seja bem vindoa ao grupo direita mg...          2009   

      word_count  char_count  unique_word_ratio  avg_word_length  
1628        15.0        93.0            1.00000         5.266667  
1528         9.0        65.0            1.00000         6.333333  
1512         9.0        65.0            1.00000         6.333333  
1540        14.0        76.0            1.00000         4.500000  
77          47.0       301.0            0.93617         5.425532  


Utilizando os dados referente a postagens no Telegram, crie um modelo preditivo (classificador multiclasse) para classificar uma mensagem em níveis de viralidade. Escolha uma estratégia para definir o número de níveis de viralidade (classes). Por exemplo, você definir quatro níveis de viralidade a partir dos quartis da quantidade de compartilhamentos.

A avaliação experimental deverá considerar:

e. O algoritmo de classificação: regressão logística, árvore de decisão e uma estratégia baseada em “ensemble”;

f. Regularização: Com regularização (Ridge, Lasso ou ElasticNet) e sem regularização;

g. Normalização dos dados: sem normalização, Z-Score, Min-Max (OPCIONAL); 

h. Pré-processamento de dados: sem pré-processamento e com pré-processamento;

i. Embedding: BOW, TF-IDF, Word2Vec;

j. N-Gramas: unigramas, bigramas, trigramas;

k. Treinamento, Validação e Teste: Outer K-Fold Cross-Validation;

In [ ]:
print("Definindo os níveis de viralidade (multiclasse) com base nos quartis...")

niveis, bin_edges = pd.qcut(result_df['shared_count'], q=4, labels=False, retbins=True, duplicates='drop')
result_df['viralidade_level'] = niveis


label_names = {0: 'Baixa', 1: 'Média', 2: 'Alta', 3: 'Viral'}
print("\nLimites de compartilhamento para cada nível de viralidade:")
for i in range(len(bin_edges) - 1):
    print(f"  - Nível {i} ({label_names[i]}): de {int(bin_edges[i])} a {int(bin_edges[i+1])} compartilhamentos")

print("\nDistribuição das Classes:")
print(result_df['viralidade_level'].value_counts().sort_index().rename(index=label_names))

y = result_df['viralidade_level']

Definindo os níveis de viralidade (multiclasse) com base nos quartis...

Limites de compartilhamento para cada nível de viralidade:
  - Nível 0 (Baixa): de 1 a 17422 compartilhamentos

Distribuição das Classes:
viralidade_level
Baixa    183796
Name: count, dtype: int64


In [13]:
print(result_df['shared_count'].describe())

porcentagem_de_uns = (result_df['shared_count'] == 1).mean() * 100
print(f"\nPorcentagem de mensagens com 'shared_count' == 1: {porcentagem_de_uns:.2f}%")

count    183796.000000
mean          2.151026
std          47.949450
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max       17422.000000
Name: shared_count, dtype: float64

Porcentagem de mensagens com 'shared_count' == 1: 82.65%


In [ ]:
print("Definindo os níveis de viralidade (multiclasse) com base em limites manuais...")

bins = [
    0,
    1, # Limite para a classe 0 (só o número 1)
    10, # Limite para a classe 1 (de 2 a 10)
    100, # Limite para a classe 2 (de 11 a 100)
    np.inf # Limite para a classe 3 (acima de 100)
]

label_names = [
    'Instância Única (0)',
    'Baixa Viralidade (1)',
    'Média Viralidade (2)',
    'Alta Viralidade (3)'
]

result_df['viralidade_level'] = pd.cut(
    result_df['shared_count'], 
    bins=bins, 
    labels=False,
    right=True
)

print("\nNova distribuição de classes (manual):")
print(result_df['viralidade_level'].value_counts().sort_index().rename(index={i: v for i, v in enumerate(label_names)}))

y = result_df['viralidade_level']

Definindo os níveis de viralidade (multiclasse) com base em limites manuais...

Nova distribuição de classes (manual):
viralidade_level
Instância Única (0)     151913
Baixa Viralidade (1)     29334
Média Viralidade (2)      2368
Alta Viralidade (3)        181
Name: count, dtype: int64


# e. O algoritmo de classificação: regressão logística, árvore de decisão e uma estratégia baseada em “ensemble”;

In [ ]:
numeric_features = [
    'word_count', 
    'char_count', 
    'avg_word_length',
    'unique_word_ratio',
]
X = result_df[numeric_features]
y = result_df['viralidade_level']

label_names = [
    'Instância Única (0)',
    'Baixa Viralidade (1)',
    'Média Viralidade (2)',
    'Alta Viralidade (3)'
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Dados divididos em {len(X_train)} para treino e {len(X_test)} para teste.")


models = {
    "Regressão Logística": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest (Ensemble)": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    print(f"\n--- Treinando e Avaliando: {name} ---")
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    print(classification_report(y_test, y_pred, target_names=label_names))

Dados divididos em 137847 para treino e 45949 para teste.

--- Treinando e Avaliando: Regressão Logística ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                      precision    recall  f1-score   support

 Instância Única (0)       0.83      1.00      0.91     38039
Baixa Viralidade (1)       0.33      0.01      0.01      7258
Média Viralidade (2)       0.00      0.00      0.00       602
 Alta Viralidade (3)       0.00      0.00      0.00        50

            accuracy                           0.83     45949
           macro avg       0.29      0.25      0.23     45949
        weighted avg       0.74      0.83      0.75     45949


--- Treinando e Avaliando: Árvore de Decisão ---
                      precision    recall  f1-score   support

 Instância Única (0)       0.84      0.94      0.89     38039
Baixa Viralidade (1)       0.25      0.09      0.13      7258
Média Viralidade (2)       0.14      0.06      0.08       602
 Alta Viralidade (3)       0.29      0.10      0.15        50

            accuracy                           0.80     45949
           macro avg       0.38      0.30      0.31     45949
        weighte

# f. Regularização: Com regularização (Ridge, Lasso ou ElasticNet) e sem regularização;

In [16]:
print("Definindo os modelos para o experimento de regularização...")

models = {
    "Reg. Log. - Sem Regularização": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(penalty=None, solver='saga', random_state=42, max_iter=2000))
    ]),
    
    "Reg. Log. - L2 (Ridge)": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(penalty='l2', C=1.0, solver='saga', random_state=42, max_iter=2000))
    ]),
    
    "Reg. Log. - L1 (Lasso)": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(penalty='l1', C=1.0, solver='saga', random_state=42, max_iter=2000))
    ]),
    
    "Reg. Log. - ElasticNet": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(penalty='elasticnet', C=1.0, solver='saga', l1_ratio=0.5, random_state=42, max_iter=2000))
    ]),

    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest (Ensemble)": RandomForestClassifier(random_state=42)
}


for name, model in models.items():
    print(f"\n--- Treinando e Avaliando: {name} ---")
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    print(classification_report(y_test, y_pred, target_names=label_names))

    if 'Reg. Log.' in name:
        classifier = model.named_steps['classifier']
        
        print("   -> Coeficientes (pesos) aprendidos para cada feature por classe:")
        coefs_df = pd.DataFrame(classifier.coef_, columns=numeric_features, index=label_names)
        print(coefs_df)

Definindo os modelos para o experimento de regularização...

--- Treinando e Avaliando: Reg. Log. - Sem Regularização ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set

                      precision    recall  f1-score   support

 Instância Única (0)       0.83      1.00      0.91     38039
Baixa Viralidade (1)       0.32      0.01      0.01      7258
Média Viralidade (2)       0.00      0.00      0.00       602
 Alta Viralidade (3)       0.00      0.00      0.00        50

            accuracy                           0.83     45949
           macro avg       0.29      0.25      0.23     45949
        weighted avg       0.74      0.83      0.75     45949

   -> Coeficientes (pesos) aprendidos para cada feature por classe:
                      word_count  char_count  avg_word_length  \
Instância Única (0)    -0.435206    0.344976        -0.425879   
Baixa Viralidade (1)   -0.249967    0.196498         0.036322   
Média Viralidade (2)    0.170731   -0.167991         0.195876   
Alta Viralidade (3)     0.514442   -0.373483         0.193681   

                      unique_word_ratio  
Instância Única (0)            0.053499  
Baixa Viralidade (1)   

c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                      precision    recall  f1-score   support

 Instância Única (0)       0.83      1.00      0.91     38039
Baixa Viralidade (1)       0.32      0.01      0.01      7258
Média Viralidade (2)       0.00      0.00      0.00       602
 Alta Viralidade (3)       0.00      0.00      0.00        50

            accuracy                           0.83     45949
           macro avg       0.29      0.25      0.23     45949
        weighted avg       0.74      0.83      0.75     45949

   -> Coeficientes (pesos) aprendidos para cada feature por classe:
                      word_count  char_count  avg_word_length  \
Instância Única (0)    -0.396321    0.306425        -0.424790   
Baixa Viralidade (1)   -0.211944    0.158814         0.037324   
Média Viralidade (2)    0.196882   -0.193737         0.196121   
Alta Viralidade (3)     0.411383   -0.271502         0.191345   

                      unique_word_ratio  
Instância Única (0)            0.055324  
Baixa Viralidade (1)   

c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                      precision    recall  f1-score   support

 Instância Única (0)       0.83      1.00      0.91     38039
Baixa Viralidade (1)       0.32      0.01      0.01      7258
Média Viralidade (2)       0.00      0.00      0.00       602
 Alta Viralidade (3)       0.00      0.00      0.00        50

            accuracy                           0.83     45949
           macro avg       0.29      0.25      0.23     45949
        weighted avg       0.74      0.83      0.75     45949

   -> Coeficientes (pesos) aprendidos para cada feature por classe:
                      word_count  char_count  avg_word_length  \
Instância Única (0)    -0.287327    0.180874        -0.462299   
Baixa Viralidade (1)   -0.100863    0.031266        -0.000167   
Média Viralidade (2)    0.265112   -0.278062         0.155937   
Alta Viralidade (3)     0.113793    0.000000         0.143113   

                      unique_word_ratio  
Instância Única (0)            0.059800  
Baixa Viralidade (1)   

c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                      precision    recall  f1-score   support

 Instância Única (0)       0.83      1.00      0.91     38039
Baixa Viralidade (1)       0.32      0.01      0.01      7258
Média Viralidade (2)       0.00      0.00      0.00       602
 Alta Viralidade (3)       0.00      0.00      0.00        50

            accuracy                           0.83     45949
           macro avg       0.29      0.25      0.23     45949
        weighted avg       0.74      0.83      0.75     45949

   -> Coeficientes (pesos) aprendidos para cada feature por classe:
                      word_count  char_count  avg_word_length  \
Instância Única (0)    -0.345647    0.249168        -0.462123   
Baixa Viralidade (1)   -0.160214    0.100545         0.000000   
Média Viralidade (2)    0.228842   -0.232059         0.157550   
Alta Viralidade (3)     0.272898   -0.143277         0.149175   

                      unique_word_ratio  
Instância Única (0)            0.057568  
Baixa Viralidade (1)   

# h. Pré-processamento de dados: sem pré-processamento e com pré-processamento;

In [ ]:
numeric_features = [
    'word_count', 
    'char_count', 
    'avg_word_length',
    'unique_word_ratio',
]

X = result_df[numeric_features]
y = result_df['viralidade_level']

label_names = [
    'Instância Única (0)',
    'Baixa Viralidade (1)',
    'Média Viralidade (2)',
    'Alta Viralidade (3)'
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Definição de X e y corrigida. Prosseguindo com o experimento...")


print("\n" + "="*20)
print("CENÁRIO 1: SEM PRÉ-PROCESSAMENTO NUMÉRICO")
print("="*20)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"Distribuição das classes no treino após SMOTE: \n{pd.Series(y_train_smote).value_counts().sort_index()}")

models_sem_preproc = {
    "Regressão Logística": LogisticRegression(random_state=42, max_iter=2000),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models_sem_preproc.items():
    print(f"\n--- Avaliando: {name} ---")
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))



print("\n" + "="*20)
print("CENÁRIO 2: COM PRÉ-PROCESSAMENTO NUMÉRICO")
print("="*20)

def remove_outliers_iqr(X, y):
    X_clean = X.copy()
    y_clean = y.copy()
    for col in X_clean.columns:
        Q1 = X_clean[col].quantile(0.3)
        Q3 = X_clean[col].quantile(0.7)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        mask = (X_clean[col] >= lower_bound) & (X_clean[col] <= upper_bound)
        X_clean = X_clean.loc[mask]
        y_clean = y_clean.loc[mask]
    return X_clean, y_clean

X_train_no_outliers, y_train_no_outliers = remove_outliers_iqr(X_train, y_train)
print(f"\nRemovendo outliers do treino: {len(X_train)} -> {len(X_train_no_outliers)} amostras")

X_train_proc_smote, y_train_proc_smote = smote.fit_resample(X_train_no_outliers, y_train_no_outliers)
print(f"Distribuição das classes no treino pré-processado após SMOTE: \n{pd.Series(y_train_proc_smote).value_counts().sort_index()}")

pipelines_com_preproc = {
    "Regressão Logística": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=2000))
    ]),
    "Árvore de Decisão": Pipeline([ 
        ('classifier', DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ('classifier', RandomForestClassifier(random_state=42))
    ])
}

for name, pipeline in pipelines_com_preproc.items():
    print(f"\n--- Avaliando: {name} ---")
    pipeline.fit(X_train_proc_smote, y_train_proc_smote)
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))

Definição de X e y corrigida. Prosseguindo com o experimento...

CENÁRIO 1: SEM PRÉ-PROCESSAMENTO NUMÉRICO
Distribuição das classes no treino após SMOTE: 
viralidade_level
0    113874
1    113874
2    113874
3    113874
Name: count, dtype: int64

--- Avaliando: Regressão Logística ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                      precision    recall  f1-score   support

 Instância Única (0)       0.88      0.63      0.73     38039
Baixa Viralidade (1)       0.19      0.12      0.14      7258
Média Viralidade (2)       0.02      0.27      0.04       602
 Alta Viralidade (3)       0.00      0.40      0.01        50

            accuracy                           0.54     45949
           macro avg       0.28      0.35      0.23     45949
        weighted avg       0.76      0.54      0.63     45949


--- Avaliando: Árvore de Decisão ---
                      precision    recall  f1-score   support

 Instância Única (0)       0.86      0.76      0.80     38039
Baixa Viralidade (1)       0.22      0.32      0.26      7258
Média Viralidade (2)       0.09      0.20      0.13       602
 Alta Viralidade (3)       0.04      0.36      0.06        50

            accuracy                           0.68     45949
           macro avg       0.30      0.41      0.31     45949
        weighted avg       

# i. Embedding: BOW, TF-IDF, Word2Vec;

In [ ]:
text_feature = 'cleaned_text'
numeric_features = ['word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']

X = result_df[numeric_features + [text_feature]]
y = result_df['viralidade_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

label_names = [
    'Instância Única (0)',
    'Baixa Viralidade (1)',
    'Média Viralidade (2)',
    'Alta Viralidade (3)'
]

models_to_run = {
    "Regressão Logística": LogisticRegression(penalty='l2', C=1.0, solver='saga', random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1)
}


print("\n" + "="*60)
print("INICIANDO CENÁRIO 1: TF-IDF")
print("="*60)

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_text_tfidf = tfidf_vectorizer.fit_transform(X_train[text_feature])
X_test_text_tfidf = tfidf_vectorizer.transform(X_test[text_feature])

scaler_tfidf = StandardScaler()
X_train_num_scaled_tfidf = scaler_tfidf.fit_transform(X_train[numeric_features])
X_test_num_scaled_tfidf = scaler_tfidf.transform(X_test[numeric_features])

X_train_final_tfidf = hstack([X_train_num_scaled_tfidf, X_train_text_tfidf])
X_test_final_tfidf = hstack([X_test_num_scaled_tfidf, X_test_text_tfidf])

print("Aplicando SMOTE para TF-IDF...")
smote_tfidf = SMOTE(random_state=42)
X_train_resampled_tfidf, y_train_resampled_tfidf = smote_tfidf.fit_resample(X_train_final_tfidf, y_train)
print("SMOTE concluído.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com TF-IDF"):
    print(f"\n--- Treinando e Avaliando: {name} ---")
    model.fit(X_train_resampled_tfidf, y_train_resampled_tfidf)
    y_pred = model.predict(X_test_final_tfidf)
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))


print("\n" + "="*60)
print("INICIANDO CENÁRIO 2: BAG-OF-WORDS (BOW)")
print("="*60)

bow_vectorizer = CountVectorizer(max_features=5000)
X_train_text_bow = bow_vectorizer.fit_transform(X_train[text_feature])
X_test_text_bow = bow_vectorizer.transform(X_test[text_feature])

scaler_bow = StandardScaler()
X_train_num_scaled_bow = scaler_bow.fit_transform(X_train[numeric_features])
X_test_num_scaled_bow = scaler_bow.transform(X_test[numeric_features])

X_train_final_bow = hstack([X_train_num_scaled_bow, X_train_text_bow])
X_test_final_bow = hstack([X_test_num_scaled_bow, X_test_text_bow])

print("Aplicando SMOTE para BOW...")
smote_bow = SMOTE(random_state=42)
X_train_resampled_bow, y_train_resampled_bow = smote_bow.fit_resample(X_train_final_bow, y_train)
print("SMOTE concluído.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com BOW"):
    print(f"\n--- Treinando e Avaliando: {name} ---")
    model.fit(X_train_resampled_bow, y_train_resampled_bow)
    y_pred = model.predict(X_test_final_bow)
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))



print("\n" + "="*60)
print("INICIANDO CENÁRIO 3: WORD2VEC")
print("="*60)

tokenized_text_train = [text.split() for text in X_train[text_feature]]
vector_size = 100
print("Treinando modelo Word2Vec...")
w2v_model = Word2Vec(sentences=tokenized_text_train, vector_size=vector_size, window=5, min_count=2, workers=4)
print("Modelo Word2Vec treinado.")

def document_vector(doc, model, num_features):
    doc_vector = np.zeros((num_features,), dtype="float32")
    num_words = 0
    words = doc.split()
    for word in words:
        if word in model.wv:
            num_words += 1
            doc_vector = np.add(doc_vector, model.wv[word])
    if num_words > 0:
        doc_vector = np.divide(doc_vector, num_words)
    return doc_vector

X_train_text_w2v = np.array([document_vector(doc, w2v_model, vector_size) for doc in tqdm(X_train[text_feature], desc="Criando vetores de treino (W2V)")])
X_test_text_w2v = np.array([document_vector(doc, w2v_model, vector_size) for doc in tqdm(X_test[text_feature], desc="Criando vetores de teste (W2V)")])

scaler_w2v = StandardScaler()
X_train_num_scaled_w2v = scaler_w2v.fit_transform(X_train[numeric_features])
X_test_num_scaled_w2v = scaler_w2v.transform(X_test[numeric_features])

X_train_final_w2v = np.hstack([X_train_num_scaled_w2v, X_train_text_w2v])
X_test_final_w2v = np.hstack([X_test_num_scaled_w2v, X_test_text_w2v])

print("Aplicando SMOTE para Word2Vec...")
smote_w2v = SMOTE(random_state=42)
X_train_resampled_w2v, y_train_resampled_w2v = smote_w2v.fit_resample(X_train_final_w2v, y_train)
print("SMOTE concluído.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com Word2Vec"):
    print(f"\n--- Treinando e Avaliando: {name} ---")
    model.fit(X_train_resampled_w2v, y_train_resampled_w2v)
    y_pred = model.predict(X_test_final_w2v)
    print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))


INICIANDO CENÁRIO 1: TF-IDF
Aplicando SMOTE para TF-IDF...
SMOTE concluído.


Treinando Modelos com TF-IDF:   0%|          | 0/2 [00:00<?, ?it/s]


--- Treinando e Avaliando: Regressão Logística ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Treinando Modelos com TF-IDF:  50%|█████     | 1/2 [17:32<17:32, 1052.58s/it]

                      precision    recall  f1-score   support

 Instância Única (0)       0.92      0.73      0.81     37978
Baixa Viralidade (1)       0.31      0.54      0.39      7334
Média Viralidade (2)       0.09      0.38      0.14       592
 Alta Viralidade (3)       0.06      0.42      0.10        45

            accuracy                           0.69     45949
           macro avg       0.34      0.52      0.36     45949
        weighted avg       0.81      0.69      0.74     45949


--- Treinando e Avaliando: Random Forest ---


...


--- Treinando e Avaliando: Regressão Logística ---


Treinando Modelos com Word2Vec:  50%|█████     | 1/2 [03:07<03:07, 187.27s/it]

                      precision    recall  f1-score   support

 Instância Única (0)       0.92      0.66      0.77     37978
Baixa Viralidade (1)       0.28      0.45      0.35      7334
Média Viralidade (2)       0.05      0.37      0.09       592
 Alta Viralidade (3)       0.01      0.71      0.03        45

            accuracy                           0.62     45949
           macro avg       0.32      0.55      0.31     45949
        weighted avg       0.81      0.62      0.69     45949


--- Treinando e Avaliando: Random Forest ---


Treinando Modelos com Word2Vec: 100%|██████████| 2/2 [05:01<00:00, 150.70s/it]

                      precision    recall  f1-score   support

 Instância Única (0)       0.89      0.86      0.88     37978
Baixa Viralidade (1)       0.37      0.43      0.40      7334
Média Viralidade (2)       0.39      0.26      0.31       592
 Alta Viralidade (3)       0.52      0.29      0.37        45

            accuracy                           0.79     45949
           macro avg       0.54      0.46      0.49     45949
        weighted avg       0.80      0.79      0.79     45949



# j. N-Gramas: unigramas, bigramas, trigramas;

In [ ]:
ngram_configs = {
    "Apenas Unigramas": (1, 1),
    "Unigramas e Bigramas": (1, 2),
    "Unigramas, Bigramas e Trigramas": (1, 3)
}


X = result_df[numeric_features + [text_feature]]
y = result_df['viralidade_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


for config_name, ngram_range in ngram_configs.items():
    print("\n" + "="*60)
    print(f"INICIANDO CENÁRIO COM TF-IDF E N-GRAMAS: {config_name}")
    print("="*60)

    tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=ngram_range)
    X_train_text = tfidf_vectorizer.fit_transform(X_train[text_feature])
    X_test_text = tfidf_vectorizer.transform(X_test[text_feature])

    scaler = StandardScaler()
    X_train_num_scaled = scaler.fit_transform(X_train[numeric_features])
    X_test_num_scaled = scaler.transform(X_test[numeric_features])

    X_train_final = hstack([X_train_num_scaled, X_train_text])
    X_test_final = hstack([X_test_num_scaled, X_test_text])
    
    print("Aplicando SMOTE...")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_final, y_train)
    print("SMOTE concluído.")

    for model_name, model in tqdm(models_to_run.items(), desc=f"Treinando Modelos ({config_name})"):
        print(f"\n--- Treinando e Avaliando: {model_name} ---")
        model.fit(X_train_resampled, y_train_resampled)
        y_pred = model.predict(X_test_final)
        print(classification_report(y_test, y_pred, target_names=label_names, zero_division=0))


INICIANDO CENÁRIO COM TF-IDF E N-GRAMAS: Apenas Unigramas
Aplicando SMOTE...
SMOTE concluído.


Treinando Modelos (Apenas Unigramas):   0%|          | 0/2 [00:00<?, ?it/s]


--- Treinando e Avaliando: Regressão Logística ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Treinando Modelos (Apenas Unigramas):  50%|█████     | 1/2 [18:46<18:46, 1126.57s/it]

                      precision    recall  f1-score   support

 Instância Única (0)       0.92      0.73      0.81     37978
Baixa Viralidade (1)       0.31      0.54      0.39      7334
Média Viralidade (2)       0.09      0.38      0.14       592
 Alta Viralidade (3)       0.06      0.42      0.10        45

            accuracy                           0.69     45949
           macro avg       0.34      0.52      0.36     45949
        weighted avg       0.81      0.69      0.74     45949


--- Treinando e Avaliando: Random Forest ---


...


--- Treinando e Avaliando: Regressão Logística ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Treinando Modelos (Unigramas, Bigramas e Trigramas):  50%|█████     | 1/2 [21:06<21:06, 1266.87s/it]

                      precision    recall  f1-score   support

 Instância Única (0)       0.92      0.73      0.81     37978
Baixa Viralidade (1)       0.31      0.54      0.39      7334
Média Viralidade (2)       0.08      0.38      0.14       592
 Alta Viralidade (3)       0.06      0.47      0.10        45

            accuracy                           0.69     45949
           macro avg       0.34      0.53      0.36     45949
        weighted avg       0.81      0.69      0.73     45949


--- Treinando e Avaliando: Random Forest ---


Treinando Modelos (Unigramas, Bigramas e Trigramas): 100%|██████████| 2/2 [33:26<00:00, 1003.48s/it]

                      precision    recall  f1-score   support

 Instância Única (0)       0.87      0.93      0.90     37978
Baixa Viralidade (1)       0.44      0.30      0.36      7334
Média Viralidade (2)       0.38      0.25      0.30       592
 Alta Viralidade (3)       0.17      0.29      0.21        45

            accuracy                           0.82     45949
           macro avg       0.47      0.44      0.44     45949
        weighted avg       0.80      0.82      0.80     45949



# k. Treinamento, Validação e Teste: Outer K-Fold Cross-Validation;

In [ ]:
print("--- INICIANDO ITEM (k): VALIDAÇÃO CRUZADA ANINHADA ---")

text_feature = 'cleaned_text'
numeric_features = ['word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']
X = result_df[numeric_features + [text_feature]]
y = result_df['viralidade_level']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('text', TfidfVectorizer(ngram_range=(1, 2), max_features=5000), text_feature)
    ],
    remainder='drop'
)

full_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(solver='saga', random_state=42, max_iter=5000))
])

param_grid = {
    'classifier__C': [0.1, 1.0, 10]
}

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

clf = GridSearchCV(
    estimator=full_pipeline, 
    param_grid=param_grid, 
    cv=inner_cv,
    scoring='f1_macro',
    n_jobs=-1
)

outer_scores = []

for i, (train_idx, test_idx) in enumerate(tqdm(outer_cv.split(X, y), total=outer_cv.get_n_splits(), desc="Outer CV Folds")):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    clf.fit(X_train, y_train)
    
    best_model = clf.best_estimator_
    y_pred = best_model.predict(X_test)
    score = f1_score(y_test, y_pred, average='macro')
    outer_scores.append(score)
    
    print(f"\nFold Externo {i+1}: Melhor C={clf.best_params_['classifier__C']}, Macro F1-Score = {score:.4f}")

print("\n" + "="*50)
print("Resultado Final da Validação Cruzada Aninhada")
print("="*50)
print(f"Scores Macro F1 em cada Fold Externo: {np.round(outer_scores, 4)}")
print(f"Média do Macro F1-Score: {np.mean(outer_scores):.4f}")
print(f"Desvio Padrão do Macro F1-Score: {np.std(outer_scores):.4f}")

--- INICIANDO ITEM (k): VALIDAÇÃO CRUZADA ANINHADA ---


Outer CV Folds:  20%|██        | 1/5 [47:07<3:08:30, 2827.69s/it]


Fold Externo 1: Melhor C=0.1, Macro F1-Score = 0.3662
